In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 06 — Journal-adjusted early article impact: LFCR and LFC

**Data unit:** paper; binary outcome is relative `HighImpact` vs `LowImpact` (journal/year bins, middle excluded).  **Milestones:** `mil1`–`mil4` are cumulative and stored in separate frozen snapshots.  `M1`–`M5` refer **only to model names**.

- Original **M4 (continuous LFCR)** is retained as the prespecified primary specification, with the same modelling code and original output filenames.
- **M5 (binary LFC presence)** is an alternative specification. It is evaluated independently, not entered alongside LFCR because `LFC = 1{LFCR>0}`.
- Added: adjusted M5 OR/95% CI, M3→M4 and M3→M5 matched-sample nested LR tests with BH within scenario, LFC observed rates, forest plot, M5 SHAP, predictive M3/M4/M5 comparison and expanded cross-milestone CSV.
- All graphics retain 11 pt text; saved as PNG (300 dpi) and PDF.

**Critical missingness caveat:** in the full dataset, papers without extracted citation-function data are encoded LFCR=0/LFC=0 **for modelling only**, with a separate `has_valid_citation_data` indicator. These papers must **not** be labelled as observed absence of LFC in descriptive comparisons. Valid-only descriptive LFC comparisons use only genuinely observed LFC=0/1. Full M5's LFC coefficient is interpreted conditionally on citation-data availability and the other controls.

**Snapshot safety:** for an already-created milestone, the three source inputs (`clean_dataset_original.csv`, `rare_citations_summary.csv`, `milestone_sampling_plan.csv`) come from that milestone's `input_files`, not the current central rare summary. Original source snapshots are not overwritten; generated analysis outputs may be regenerated on rerun.


In [ ]:
# =========================
# 1. SETUP
# =========================
import os
import shutil
import hashlib
import re
from datetime import datetime

import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

import statsmodels.api as sm
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests

BASE_DIR = '/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER'
MILESTONES_BASE_DIR = os.path.join(BASE_DIR, "milestones_journal_level")

os.makedirs(MILESTONES_BASE_DIR, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("MILESTONES_BASE_DIR:", MILESTONES_BASE_DIR)
# =========================
# PUBLICATION-READY FIGURE SETTINGS
# =========================
#FIGSIZE_MAIN = (8.2, 5.2)
#FIGSIZE_WIDE = (9.2, 5.6)
#FIGSIZE_HEATMAP = (8.4, 5.2)
FIGSIZE_MAIN = (6, 4)
FIGSIZE_WIDE = (9, 5)
FIGSIZE_HEATMAP = (7, 5)

FIG_DPI = 300
FIG_FONT_SIZE = 11

# Main-paper plots focus on the substantive continuous predictors.
# Journal fixed effects and availability controls remain in the fitted models,
# but are not displayed in the main SHAP figures because they are adjustment
# variables rather than the focal explanatory variables.
PLOT_FOCAL_FEATURES = [
    "less_frequent_citation_ratio",
    "references",
    "year",
]
# Parallel M5 figures show the alternative LFC operationalization, not LFCR.
PLOT_LFC_FEATURES = [
    "has_less_frequent_citation",
    "references",
    "year",
]

sns.set_theme(style="whitegrid", context="paper")

BAR_COLOR = "#5975A4"

plt.rcParams.update({
    "font.size": FIG_FONT_SIZE,
    "axes.titlesize": FIG_FONT_SIZE,
    "axes.labelsize": FIG_FONT_SIZE,
    "xtick.labelsize": FIG_FONT_SIZE,
    "ytick.labelsize": FIG_FONT_SIZE,
    "legend.fontsize": FIG_FONT_SIZE,
    "figure.titlesize": FIG_FONT_SIZE,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


def apply_figure_fontsize(fig, fontsize=FIG_FONT_SIZE):
    """Force every visible text element in a Matplotlib/Seaborn/SHAP figure to one size."""
    for ax in fig.axes:
        ax.title.set_fontsize(fontsize)
        ax.xaxis.label.set_fontsize(fontsize)
        ax.yaxis.label.set_fontsize(fontsize)
        ax.tick_params(axis="both", labelsize=fontsize)

        legend = ax.get_legend()
        if legend is not None:
            for txt in legend.get_texts():
                txt.set_fontsize(fontsize)
            if legend.get_title() is not None:
                legend.get_title().set_fontsize(fontsize)

        # Includes heatmap annotations and SHAP auxiliary text when present.
        for txt in ax.texts:
            txt.set_fontsize(fontsize)

    for txt in fig.texts:
        txt.set_fontsize(fontsize)

FEATURE_LABEL_MAP = {
    "less_frequent_citation_ratio": "LFCR",
    "has_less_frequent_citation": "LFC",
    "has_valid_citation_data": "Valid citation data",
    "references": "References",
    "year": "Year",
    "citations": "Citations",
}

def get_journal_short_map(data):
    if {"journal", "journal_short"}.issubset(data.columns):
        return (
            data[["journal", "journal_short"]]
            .dropna()
            .drop_duplicates()
            .set_index("journal")["journal_short"]
            .astype(str)
            .to_dict()
        )
    return {}

def clean_feature_name(name, journal_short_map=None):
    name = str(name)
    journal_short_map = journal_short_map or {}
    if name.startswith("journal_"):
        full_journal = name[len("journal_"):]
        return journal_short_map.get(full_journal, full_journal)
    return FEATURE_LABEL_MAP.get(name, name.replace("_", " "))

def save_figure(base_path):
    plt.savefig(base_path + ".png", dpi=FIG_DPI, bbox_inches="tight")
    plt.savefig(base_path + ".pdf", bbox_inches="tight")

# =========================
# 2. MILESTONE SETUP
# =========================

In [ ]:
print("Existing journal-level milestones:")
existing = sorted([
    d for d in os.listdir(MILESTONES_BASE_DIR)
    if os.path.isdir(os.path.join(MILESTONES_BASE_DIR, d))
])

for i, name in enumerate(existing, start=1):
    print(f"{i}. {name}")

# Input files in BASE_DIR
base_clean = os.path.join(BASE_DIR, "clean_dataset.csv")
base_rare = os.path.join(BASE_DIR, "rare_citations_summary.csv")

# The 03_coverage_sampling_plan.py script saves the main plan here:
# BASE_DIR/coverage_sampling_plan/milestone_sampling_plan.csv
base_sampling_plan = os.path.join(
    BASE_DIR,
    "coverage_sampling_plan",
    "milestone_sampling_plan.csv"
)

# This is the exact file created by 03_coverage_sampling_plan.py.
# It is NOT named coverage_sampling_plan.csv; it is inside the coverage_sampling_plan folder.
if not os.path.exists(base_sampling_plan):
    raise FileNotFoundError(
        "Δεν βρέθηκε το sampling plan που δημιουργεί το 03_coverage_sampling_plan.py:\n"
        f"{base_sampling_plan}\n"
        "Πρώτα τρέξε το 03_coverage_sampling_plan.py ώστε να δημιουργηθεί "
        "το coverage_sampling_plan/milestone_sampling_plan.csv."
    )

# Read sampling plan early so we can validate/select milestones before creating outputs.
sampling_plan_preview = pd.read_csv(base_sampling_plan)

required_sampling_cols_preview = ["filename", "add_in_milestone"]
missing_sampling_preview = [c for c in required_sampling_cols_preview if c not in sampling_plan_preview.columns]

if missing_sampling_preview:
    raise ValueError(
        f"Missing required columns from sampling plan: {missing_sampling_preview}. "
        "Expected at least: filename, add_in_milestone"
    )

# If milestone_order does not exist, infer it from names like v1_coverage_30pct.
def infer_milestone_order(name):
    match = re.match(r"v(\d+)_", str(name).strip())
    if match is None:
        raise ValueError(
            f"Cannot infer milestone order from: {name}. "
            "Use names like v1_coverage_30pct, v2_coverage_50pct, etc., "
            "or include a milestone_order column in the sampling plan."
        )
    return int(match.group(1))

if "milestone_order" not in sampling_plan_preview.columns:
    sampling_plan_preview["milestone_order"] = sampling_plan_preview["add_in_milestone"].apply(infer_milestone_order)

available_milestones_df = (
    sampling_plan_preview[["add_in_milestone", "milestone_order"]]
    .drop_duplicates()
    .sort_values("milestone_order")
)

available_milestones = available_milestones_df["add_in_milestone"].tolist()

print("\nAvailable milestones from sampling plan:")
for i, milestone_name in enumerate(available_milestones, start=1):
    order = available_milestones_df.loc[
        available_milestones_df["add_in_milestone"] == milestone_name,
        "milestone_order"
    ].iloc[0]
    print(f"{i}. {milestone_name} (order={order})")

user_input = input(
    "\nΔώσε milestone από τα παραπάνω, π.χ. v1_coverage_30pct, "
    "ή αριθμό επιλογής 1-4: "
).strip()

if user_input.isdigit():
    choice_idx = int(user_input) - 1
    if choice_idx < 0 or choice_idx >= len(available_milestones):
        raise ValueError(f"Invalid milestone choice: {user_input}")
    MILESTONE = available_milestones[choice_idx]
elif user_input:
    MILESTONE = user_input
else:
    # Safe default: first milestone, not a timestamped run, because cumulative filtering requires a known milestone.
    MILESTONE = available_milestones[0]

if MILESTONE not in available_milestones:
    raise ValueError(
        f"Το milestone '{MILESTONE}' δεν υπάρχει στο sampling plan. "
        f"Available milestones: {available_milestones}"
    )

CURRENT_MILESTONE_ORDER = int(
    available_milestones_df.loc[
        available_milestones_df["add_in_milestone"] == MILESTONE,
        "milestone_order"
    ].iloc[0]
)

MILESTONE_DIR = os.path.join(MILESTONES_BASE_DIR, MILESTONE)
INPUT_SNAPSHOT_DIR = os.path.join(MILESTONE_DIR, "input_files")
OUTPUT_DIR = os.path.join(MILESTONE_DIR, "outputs")

os.makedirs(INPUT_SNAPSHOT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

snapshot_clean = os.path.join(INPUT_SNAPSHOT_DIR, "clean_dataset_original.csv")
snapshot_rare = os.path.join(INPUT_SNAPSHOT_DIR, "rare_citations_summary.csv")
snapshot_sampling_plan = os.path.join(INPUT_SNAPSHOT_DIR, "milestone_sampling_plan.csv")

if (
    os.path.exists(snapshot_clean)
    and os.path.exists(snapshot_rare)
    and os.path.exists(snapshot_sampling_plan)
):
    print("\nResume mode: loading from existing milestone snapshot")
    CLEAN_DATASET_PATH = snapshot_clean
    RARE_DATASET_PATH = snapshot_rare
    SAMPLING_PLAN_PATH = snapshot_sampling_plan
elif (
    not os.path.exists(snapshot_clean)
    and not os.path.exists(snapshot_rare)
    and not os.path.exists(snapshot_sampling_plan)
):
    print("\nNew milestone mode: loading from BASE_DIR")
    CLEAN_DATASET_PATH = base_clean
    RARE_DATASET_PATH = base_rare
    SAMPLING_PLAN_PATH = base_sampling_plan
else:
    raise FileNotFoundError(
        "Incomplete milestone snapshot. Expected all three files or none:\n"
        "- clean_dataset_original.csv\n"
        "- rare_citations_summary.csv\n"
        "- milestone_sampling_plan.csv"
    )

# Audit the three immutable source snapshots (when resume mode is active).
# All subsequent writes go to derived files/outputs, never to these source files.
SNAPSHOT_MODE = all(os.path.isfile(p) for p in (
    snapshot_clean, snapshot_rare, snapshot_sampling_plan
))
def input_sha256(path):
    hasher = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

source_paths = {
    "clean_dataset_original.csv": CLEAN_DATASET_PATH,
    "rare_citations_summary.csv": RARE_DATASET_PATH,
    "milestone_sampling_plan.csv": SAMPLING_PLAN_PATH,
}
source_hashes_before = {name: input_sha256(path) for name, path in source_paths.items()}
print("Input mode:", "FROZEN MILESTONE SNAPSHOT" if SNAPSHOT_MODE else "FIRST-RUN BASE_DIR INPUTS")
if SNAPSHOT_MODE:
    assert os.path.abspath(RARE_DATASET_PATH) == os.path.abspath(snapshot_rare), (
        "Resume mode MUST use the milestone's frozen rare_citations_summary.csv"
    )

print("Running milestone:", MILESTONE)
print("Current milestone order:", CURRENT_MILESTONE_ORDER)
print("Clean dataset:", CLEAN_DATASET_PATH)
print("Rare summary:", RARE_DATASET_PATH)
print("Sampling plan:", SAMPLING_PLAN_PATH)
print("Outputs:", OUTPUT_DIR)

# =========================
# 3. LOAD + MERGE
# =========================

In [ ]:
df = pd.read_csv(CLEAN_DATASET_PATH)
rare_df = pd.read_csv(RARE_DATASET_PATH)
sampling_df = pd.read_csv(SAMPLING_PLAN_PATH)

# Snapshot input files only once.
if not os.path.exists(snapshot_clean):
    shutil.copy(CLEAN_DATASET_PATH, snapshot_clean)

if not os.path.exists(snapshot_rare):
    shutil.copy(RARE_DATASET_PATH, snapshot_rare)

if not os.path.exists(snapshot_sampling_plan):
    shutil.copy(SAMPLING_PLAN_PATH, snapshot_sampling_plan)

# Backward compatibility: αν υπάρχει παλιά ονομασία, τη μετατρέπουμε.
rename_map = {
    "rare_citation_ratio": "less_frequent_citation_ratio",
    "correct_rare_citation_ratio": "has_valid_citation_data",
    "has_rare": "has_less_frequent_citation",
    "discipline": "journal"  # fallback μόνο αν δοθεί παλιό sample
}

df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns and v not in df.columns})
rare_df = rare_df.rename(columns={k: v for k, v in rename_map.items() if k in rare_df.columns and v not in rare_df.columns})
sampling_df = sampling_df.rename(columns={k: v for k, v in rename_map.items() if k in sampling_df.columns and v not in sampling_df.columns})

required_clean_cols = ["filename", "journal", "year", "citations", "citation_bin", "references"]
missing = [c for c in required_clean_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns from clean_dataset.csv: {missing}")

if "less_frequent_citation_ratio" not in rare_df.columns:
    raise ValueError("rare_citations_summary.csv must include less_frequent_citation_ratio")

# Always take less_frequent_citation_ratio from external rare_citations_summary.csv.
if "less_frequent_citation_ratio" in df.columns:
    df = df.drop(columns=["less_frequent_citation_ratio"])

df["filename"] = df["filename"].astype(str).str.strip()
rare_df["filename"] = rare_df["filename"].astype(str).str.strip()

df = df.merge(
    rare_df[["filename", "less_frequent_citation_ratio"]],
    on="filename",
    how="left"
)

print("Merged dataset shape:", df.shape)
display(df.head())

# =========================
# 3b. CUMULATIVE MILESTONE FILTERING
# =========================

In [ ]:
# The sampling plan is produced by 03_coverage_sampling_plan.py.
# It contains one row per paper and the milestone where the paper is first added.
required_sampling_cols = ["filename", "add_in_milestone"]
missing_sampling = [c for c in required_sampling_cols if c not in sampling_df.columns]

if missing_sampling:
    raise ValueError(
        f"Missing required columns from sampling plan: {missing_sampling}. "
        "Expected columns: filename, add_in_milestone"
    )

sampling_df["filename"] = sampling_df["filename"].astype(str).str.strip()
sampling_df["add_in_milestone"] = sampling_df["add_in_milestone"].astype(str).str.strip()

if "milestone_order" not in sampling_df.columns:
    sampling_df["milestone_order"] = sampling_df["add_in_milestone"].apply(infer_milestone_order)

sampling_df["milestone_order"] = pd.to_numeric(sampling_df["milestone_order"], errors="coerce")

if sampling_df["milestone_order"].isna().any():
    bad_rows = sampling_df[sampling_df["milestone_order"].isna()]["add_in_milestone"].unique()
    raise ValueError(f"Some sampling-plan rows have invalid milestone_order: {bad_rows}")

# Keep only papers that are eligible up to the current milestone.
# Example: v2 includes rows first added in v1 and rows first added in v2.
eligible_plan = sampling_df[sampling_df["milestone_order"] <= CURRENT_MILESTONE_ORDER].copy()

# Safety: each filename should appear only once in the sampling plan.
duplicate_files = sampling_df[sampling_df["filename"].duplicated(keep=False)]["filename"].unique()
if len(duplicate_files) > 0:
    raise ValueError(
        "Duplicate filenames found in sampling plan. A paper should be assigned "
        f"to exactly one add_in_milestone. Examples: {duplicate_files[:10]}"
    )

before_filter_n = len(df)

# Merge the milestone metadata into df; inner join performs the cumulative filtering.
cols_to_add = ["filename", "add_in_milestone", "milestone_order"]
for optional_col in ["milestone_pct"]:
    if optional_col in eligible_plan.columns:
        cols_to_add.append(optional_col)

eligible_plan_small = eligible_plan[cols_to_add].drop_duplicates("filename")

df = df.merge(
    eligible_plan_small,
    on="filename",
    how="inner"
)

after_filter_n = len(df)
removed_future_rows = before_filter_n - after_filter_n

print("\nCumulative milestone filtering applied")
print("Current milestone:", MILESTONE)
print("Current milestone order:", CURRENT_MILESTONE_ORDER)
print("Rows before cumulative milestone filtering:", before_filter_n)
print("Rows after cumulative milestone filtering:", after_filter_n)
print("Removed future-milestone/non-planned rows:", removed_future_rows)

print("\nIncluded add_in_milestone values:")
print(df["add_in_milestone"].value_counts().sort_index())

# Save the exact eligible file list used by this milestone for auditability.
eligible_filelist_path = os.path.join(
    INPUT_SNAPSHOT_DIR,
    f"{MILESTONE}_cumulative_eligible_filelist_used.txt"
)

pd.Series(sorted(df["filename"].unique())).to_csv(
    eligible_filelist_path,
    index=False,
    header=False
)

print("\nSaved cumulative eligible filelist used:")
print(eligible_filelist_path)

display(df.head())

# =========================
# 4. FEATURE ENGINEERING
# =========================

In [ ]:
# Availability flag
df["has_valid_citation_data"] = df["less_frequent_citation_ratio"].notna().astype(int)

# Binary presence feature
df["has_less_frequent_citation"] = (df["less_frequent_citation_ratio"] > 0).astype(int)

# Optional numeric cleanup
for col in ["citations", "references", "year", "less_frequent_citation_ratio"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Binary target: low vs high. Mid removed.
df = df[df["citation_bin"].isin(["low", "high"])].copy()
df["label"] = (df["citation_bin"] == "high").astype(int)

print("Dataset size after low/high filtering:", len(df))
print("\nCitation bin counts:")
print(df["citation_bin"].value_counts())

print("\nJournal counts:")
print(df.groupby("journal").size().sort_values(ascending=False))

print("\nJournal x citation_bin counts:")
display(pd.crosstab(df["journal"], df["citation_bin"]))

coverage = df["has_valid_citation_data"].mean()
print("\nCoverage:", coverage)

# Ensure a short journal label exists for plots/tables.
if "journal_short" not in df.columns:
    df["journal_short"] = df["journal"].astype(str)

# Save enriched snapshot
df.to_csv(os.path.join(INPUT_SNAPSHOT_DIR, "clean_dataset_enriched.csv"), index=False)

# =========================
# 5. AVAILABILITY-BIAS DESCRIPTIVE TABLES
# =========================

In [ ]:
availability_by_journal = (
    df.groupby("journal")
    .agg(
        N=("filename", "count"),
        valid_N=("has_valid_citation_data", "sum"),
        coverage=("has_valid_citation_data", "mean"),
        high_rate=("label", "mean"),
        mean_citations_3yr=("citations", "mean"),
        median_citations_3yr=("citations", "median"),
        mean_references=("references", "mean")
    )
    .reset_index()
    .sort_values("N", ascending=False)
)

availability_by_journal.to_csv(
    os.path.join(OUTPUT_DIR, "availability_bias_by_journal.csv"),
    index=False
)

availability_by_journal = availability_by_journal.merge(
    df[["journal", "journal_short"]].drop_duplicates(),
    on="journal",
    how="left"
)

availability_by_journal = availability_by_journal[
    [
        "journal_short",
        "journal",
        "N",
        "valid_N",
        "coverage",
        "high_rate",
        "mean_citations_3yr",
        "median_citations_3yr",
        "mean_references"
    ]
]

display(availability_by_journal)

# Publication-ready availability plot
fig, ax = plt.subplots(figsize=FIGSIZE_MAIN)
sns.barplot(
    data=availability_by_journal,
    x="coverage",
    y="journal_short",
    ax=ax,
    color=sns.color_palette()[0],
)
ax.set_xlabel("Coverage of valid citation data")
ax.set_ylabel("Journal")
ax.set_xlim(0, 1)
ax.grid(axis="x", alpha=0.25)
ax.grid(axis="y", visible=False)
apply_figure_fontsize(fig)
fig.tight_layout()
save_figure(os.path.join(OUTPUT_DIR, "availability_coverage_by_journal"))
plt.show()

# =========================
# 6. CREATE TWO ANALYSIS DATASETS
# =========================

In [ ]:
df_valid = df[df["has_valid_citation_data"] == 1].copy()

df_full = df.copy()
df_full["less_frequent_citation_ratio"] = df_full["less_frequent_citation_ratio"].fillna(0)

print("Valid-only dataset size:", len(df_valid))
print("Full dataset size:", len(df_full))
print("Missing less_frequent_citation_ratio rows in full dataset:", (df_full["has_valid_citation_data"] == 0).sum())

df_valid.to_csv(os.path.join(INPUT_SNAPSHOT_DIR, "clean_dataset_valid_only.csv"), index=False)
df_full.to_csv(os.path.join(INPUT_SNAPSHOT_DIR, "clean_dataset_full_with_missing_flag.csv"), index=False)

metadata = {
    "milestone": MILESTONE,
    "milestone_filtering": "cumulative_eligible_dataset",
    "current_milestone_order": int(CURRENT_MILESTONE_ORDER),
    "n_total_low_high": int(len(df)),
    "n_valid_low_high": int(df["has_valid_citation_data"].sum()),
    "n_missing_low_high": int((df["has_valid_citation_data"] == 0).sum()),
    "coverage": float(df["has_valid_citation_data"].mean()),
    "n_journals": int(df["journal"].nunique()),
    "included_add_in_milestones": "; ".join(sorted(df["add_in_milestone"].dropna().unique()))
}

pd.Series(metadata).to_csv(os.path.join(MILESTONE_DIR, "metadata.csv"))

config = {
    "milestone": MILESTONE,
    "current_milestone_order": int(CURRENT_MILESTONE_ORDER),
    "clean_dataset_path_used": CLEAN_DATASET_PATH,
    "rare_dataset_path_used": RARE_DATASET_PATH,
    "sampling_plan_path_used": SAMPLING_PLAN_PATH,
    "milestone_filtering": "cumulative_eligible_dataset",
    "final_model_valid": "M4_ratio_adjusted",
    "final_model_full": "M4_ratio_adjusted",
    "analysis_unit": "paper",
    "journal_adjustment": "categorical_fixed_effects",
    "secondary_grouping_variable": "journal",
    "random_state": 42,
    "test_size": 0.20
}

pd.Series(config).to_csv(os.path.join(MILESTONE_DIR, "config.csv"))

# =========================
# 7. PARSIMONIOUS EXPERIMENT DEFINITIONS
# =========================

The model set is intentionally compact.

Main logic:
- M1: unadjusted association of citation-function intensity (ratio).
- M2: unadjusted association of citation-function presence.
- M3: controls-only baseline (references + year + journal fixed effects; plus availability in full data).
- M4: PRIMARY adjusted ratio model.
- M5: secondary adjusted presence model.

Ratio and presence are deliberately NOT entered together in the primary model, because
`has_less_frequent_citation` is structurally derived from whether the ratio is > 0.
Treating them as alternative operationalizations keeps inference cleaner and more parsimonious.

`references` is the number of works listed in the focal paper's bibliography.
`journal` is categorical and is converted to fixed-effect dummy variables automatically.

In [ ]:
# Valid-only: all rows have valid citation-function data.
experiment_dict_valid = {
    "M1_ratio_only": ["less_frequent_citation_ratio"],
    "M2_presence_only": ["has_less_frequent_citation"],
    "M3_controls_only": ["references", "year", "journal"],
    "M4_ratio_adjusted": [
        "less_frequent_citation_ratio",
        "references",
        "year",
        "journal",
    ],
    "M5_presence_adjusted": [
        "has_less_frequent_citation",
        "references",
        "year",
        "journal",
    ],
}

# Full dataset: citation-function availability is explicitly controlled.
experiment_dict_full = {
    "M0_availability_only": ["has_valid_citation_data"],
    "M1_ratio_plus_availability": [
        "less_frequent_citation_ratio",
        "has_valid_citation_data",
    ],
    "M2_presence_plus_availability": [
        "has_less_frequent_citation",
        "has_valid_citation_data",
    ],
    "M3_controls_plus_availability": [
        "has_valid_citation_data",
        "references",
        "year",
        "journal",
    ],
    "M4_ratio_adjusted": [
        "less_frequent_citation_ratio",
        "has_valid_citation_data",
        "references",
        "year",
        "journal",
    ],
    "M5_presence_adjusted": [
        "has_less_frequent_citation",
        "has_valid_citation_data",
        "references",
        "year",
        "journal",
    ],
}

# =========================
# 8. HELPER FUNCTIONS
# =========================

In [ ]:
GROUP_COL = "journal"
MIN_GROUP_N = 100

CONTINUOUS_FEATURES = {
    "less_frequent_citation_ratio",
    "references",
    "year",
}


def has_required_columns(data, features):
    required = list(dict.fromkeys(features + ["label", GROUP_COL]))
    missing_cols = [col for col in required if col not in data.columns]
    if missing_cols:
        print(f"Missing columns: {missing_cols}")
        return False
    return True


def build_design_matrix(model_data, features):
    """Build numeric X; journal is dummy-coded as a categorical fixed effect."""
    X = model_data[features].copy()

    if GROUP_COL in X.columns:
        levels = sorted(X[GROUP_COL].dropna().astype(str).unique())
        X[GROUP_COL] = pd.Categorical(
            X[GROUP_COL].astype(str), categories=levels, ordered=False
        )
        X = pd.get_dummies(
            X,
            columns=[GROUP_COL],
            prefix=GROUP_COL,
            drop_first=True,
            dtype=float,
        )

    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")

    return X.astype(float)


def remove_zero_variance_features(X):
    zero_var = X.columns[X.nunique(dropna=False) <= 1].tolist()
    if zero_var:
        X = X.drop(columns=zero_var)
    return X, zero_var


def scale_predictive_train_test(X_train, X_test):
    """Scale only continuous predictors; binary indicators/dummies remain 0/1."""
    X_train_out = X_train.copy().astype(float)
    X_test_out = X_test.copy().astype(float)
    continuous_cols = [c for c in X_train_out.columns if c in CONTINUOUS_FEATURES]

    if continuous_cols:
        scaler = StandardScaler()
        X_train_out.loc[:, continuous_cols] = scaler.fit_transform(
            X_train_out[continuous_cols]
        )
        X_test_out.loc[:, continuous_cols] = scaler.transform(
            X_test_out[continuous_cols]
        )

    return X_train_out, X_test_out


def scale_predictive_full(X):
    """Scale only continuous predictors in a full-sample predictive/SHAP fit."""
    X_out = X.copy().astype(float)
    continuous_cols = [c for c in X_out.columns if c in CONTINUOUS_FEATURES]

    if continuous_cols:
        scaler = StandardScaler()
        X_out.loc[:, continuous_cols] = scaler.fit_transform(X_out[continuous_cols])

    return X_out


def safe_logit_fit(y, X_df):
    X_sm = sm.add_constant(X_df, has_constant="add")
    return sm.Logit(
        y.reset_index(drop=True),
        X_sm.reset_index(drop=True),
    ).fit(disp=0)


def compute_metrics(y_true, y_pred, y_score=None):
    out = {
        "Accuracy": round(accuracy_score(y_true, y_pred), 3),
        "F1": round(f1_score(y_true, y_pred, zero_division=0), 3),
        "Precision": round(precision_score(y_true, y_pred, zero_division=0), 3),
        "Recall": round(recall_score(y_true, y_pred, zero_division=0), 3),
    }
    if y_score is not None and len(np.unique(y_true)) == 2:
        try:
            out["ROC_AUC"] = round(roc_auc_score(y_true, y_score), 3)
        except Exception:
            out["ROC_AUC"] = np.nan
    return out

# =========================
# 9. RUN ANALYSIS FUNCTION
# =========================

In [ ]:
def run_analysis(data, scenario_name, experiment_dict, final_model_name):
    print(f"\n\n########################################")
    print(f"RUNNING SCENARIO: {scenario_name}")
    print(f"########################################")

    scenario_dir = os.path.join(OUTPUT_DIR, scenario_name)
    os.makedirs(scenario_dir, exist_ok=True)

    results = []
    pvalues_results = []

    for exp_name, features in experiment_dict.items():
        print(f"\n==============================")
        print(f"RUNNING {scenario_name} / {exp_name}")
        print(f"FEATURES: {features}")
        print(f"==============================")

        if not has_required_columns(data, features):
            continue

        model_data = data.dropna(subset=features + ["label"]).copy()

        if len(model_data) < 50 or model_data["label"].nunique() < 2:
            print(f"Skipping {exp_name}: insufficient usable rows/classes")
            continue

        X = build_design_matrix(model_data, features)
        y = model_data["label"].copy()
        X, zero_var = remove_zero_variance_features(X)

        if zero_var:
            print(f"Dropped zero-variance features: {zero_var}")
        if X.shape[1] == 0:
            continue

        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=0.20,
            random_state=42,
            stratify=y,
        )

        X_train, train_zero_var = remove_zero_variance_features(X_train)
        if train_zero_var:
            print(f"Dropped train-zero-variance features: {train_zero_var}")
        X_test = X_test[X_train.columns].copy()

        X_train_model, X_test_model = scale_predictive_train_test(X_train, X_test)

        model = LogisticRegression(max_iter=1000)
        model.fit(X_train_model, y_train)

        y_pred = model.predict(X_test_model)
        y_score = model.predict_proba(X_test_model)[:, 1]
        metric_dict = compute_metrics(y_test, y_pred, y_score)

        row = {
            "Scenario": scenario_name,
            "Model": exp_name,
            "Features": ", ".join(X_train_model.columns),
            "N": len(model_data),
        }
        row.update(metric_dict)
        results.append(row)

        # Diagnostic only; primary inference is the later full-sample model.
        try:
            sm_model = safe_logit_fit(y_train, X_train_model)
            for feat in sm_model.params.index:
                pvalues_results.append({
                    "Scenario": scenario_name,
                    "Model": exp_name,
                    "Feature": feat,
                    "Coefficient": round(sm_model.params[feat], 4),
                    "p-value": round(sm_model.pvalues[feat], 6),
                })
        except Exception as e:
            print(f"Statsmodels failed for {scenario_name} / {exp_name}: {e}")

    results_df = pd.DataFrame(results)
    pvalues_df = pd.DataFrame(pvalues_results)

    results_df.to_csv(
        os.path.join(scenario_dir, f"logistic_performance_results_{scenario_name}.csv"),
        index=False,
    )
    pvalues_df.to_csv(
        os.path.join(scenario_dir, f"logistic_pvalues_results_{scenario_name}.csv"),
        index=False,
    )

    display(results_df)
    display(pvalues_df)

    # -------------------------
    # Global SHAP: main adjusted model
    # -------------------------
    final_features = experiment_dict[final_model_name]
    final_data = data.dropna(subset=final_features + ["label"]).copy()

    if len(final_data) < 50 or final_data["label"].nunique() < 2:
        print(f"Skipping global SHAP for {scenario_name}: insufficient data.")
        return results_df, pvalues_df

    X_final = build_design_matrix(final_data, final_features)
    y_final = final_data["label"].copy()
    X_final, zero_var_final = remove_zero_variance_features(X_final)
    X_final_model = scale_predictive_full(X_final)

    model_final = LogisticRegression(max_iter=1000)
    model_final.fit(X_final_model, y_final)

    explainer_final = shap.Explainer(
        model_final,
        X_final_model,
        feature_names=X_final_model.columns.tolist(),
    )
    shap_values_final = explainer_final(X_final_model)

    journal_short_map = get_journal_short_map(data)

    # MAIN SHAP FIGURE: display only focal continuous predictors.
    # IMPORTANT: SHAP values still come from the FULL adjusted M4 model,
    # which continues to include journal fixed effects (and availability in
    # the full-dataset scenario). Only the visualization is filtered.
    shap_plot_features = [
        col for col in PLOT_FOCAL_FEATURES
        if col in X_final_model.columns
    ]
    if not shap_plot_features:
        raise ValueError("No focal SHAP plotting features were found in the final model.")

    shap_plot_indices = [X_final_model.columns.get_loc(col) for col in shap_plot_features]
    X_final_plot = X_final_model[shap_plot_features].copy()
    shap_values_final_plot = shap_values_final.values[:, shap_plot_indices]
    clean_plot_feature_names = [
        clean_feature_name(col, journal_short_map)
        for col in shap_plot_features
    ]

    shap.summary_plot(
        shap_values_final_plot,
        X_final_plot,
        feature_names=clean_plot_feature_names,
        show=False,
        plot_size=FIGSIZE_WIDE,
        max_display=len(shap_plot_features),
    )
    fig = plt.gcf()
    ax = plt.gca()
    ax.set_xlim(-0.6, 3.8)
    ax.set_xticks([-0.5, 0, 0.5, 1, 1.5, 2, 2.5, 3, 3.5])
    ax.set_xlabel("SHAP value (M4)")
    apply_figure_fontsize(fig)
    plt.tight_layout()
    save_figure(
        os.path.join(
            scenario_dir,
            f"shap_summary_global_final_model_{scenario_name}",
        )
    )
    plt.show()

    global_importance = np.abs(shap_values_final.values).mean(0)
    global_importance_df = pd.DataFrame({
        "Feature": X_final_model.columns,
        "MeanAbsSHAP": global_importance,
    }).sort_values(by="MeanAbsSHAP", ascending=False)

    global_importance_df.to_csv(
        os.path.join(scenario_dir, f"shap_global_importance_final_model_{scenario_name}.csv"),
        index=False,
    )
    display(global_importance_df)

    # Main-paper SHAP bar plot: same focal variables only.
    # The CSV above intentionally retains importance values for ALL fitted features.
    global_importance_plot_df = global_importance_df[
        global_importance_df["Feature"].isin(PLOT_FOCAL_FEATURES)
    ].copy()
    global_importance_plot_df["Feature_Label"] = [
        clean_feature_name(name, journal_short_map)
        for name in global_importance_plot_df["Feature"]
    ]

    fig, ax = plt.subplots(figsize=FIGSIZE_MAIN)

    sns.barplot(
        data=global_importance_plot_df,
        x="MeanAbsSHAP",
        y="Feature_Label",
        ax=ax,
        color=BAR_COLOR,
        saturation=1,
        width=0.7,
    )

    ax.set_xlabel("Mean |SHAP value| (M4)")
    ax.set_xlim(0, 0.50)
    ax.set_xticks(np.arange(0, 0.501, 0.05))
    ax.set_ylabel("")
    ax.grid(axis="x", alpha=0.25)
    ax.grid(axis="y", visible=False)
    apply_figure_fontsize(fig)
    fig.tight_layout()
    save_figure(
        os.path.join(
            scenario_dir,
            f"shap_global_barplot_final_model_{scenario_name}",
        )
    )
    plt.show()

    # -------------------------
    # Secondary within-journal analyses
    # -------------------------
    # journal is NOT a predictor here because it is constant within a journal.
    within_journal_features = [f for f in final_features if f != GROUP_COL]
    group_values = sorted(final_data[GROUP_COL].dropna().unique())

    importance_list = []
    coef_list_by_group = []
    pvalues_list_by_group = []

    for group_value in group_values:
        subset = final_data[final_data[GROUP_COL] == group_value].copy()

        if len(subset) < MIN_GROUP_N:
            print(f"Skipping {group_value} (too few samples: {len(subset)})")
            continue
        if subset["label"].nunique() < 2:
            print(f"Skipping {group_value} (only one class present)")
            continue

        X_group = build_design_matrix(subset, within_journal_features)
        y_group = subset["label"].copy()
        X_group, zero_var_group = remove_zero_variance_features(X_group)

        if zero_var_group:
            print(f"{group_value}: dropped zero-variance features: {zero_var_group}")
        if X_group.shape[1] == 0:
            continue

        X_group_model = scale_predictive_full(X_group)

        model_group = LogisticRegression(max_iter=1000)
        model_group.fit(X_group_model, y_group)

        explainer_group = shap.Explainer(
            model_group,
            X_group_model,
            feature_names=X_group_model.columns.tolist(),
        )
        shap_values_group = explainer_group(X_group_model)
        abs_shap_group = np.abs(shap_values_group.values).mean(0)

        row_importance = {GROUP_COL: group_value, "N": len(subset)}
        for i, feat in enumerate(X_group_model.columns):
            row_importance[feat] = abs_shap_group[i]
        importance_list.append(row_importance)

        row_coef = {GROUP_COL: group_value, "N": len(subset)}
        for i, feat in enumerate(X_group_model.columns):
            row_coef[feat] = round(model_group.coef_[0][i], 4)
        coef_list_by_group.append(row_coef)

        try:
            sm_group_model = safe_logit_fit(y_group, X_group_model)
            for feat in sm_group_model.params.index:
                pvalues_list_by_group.append({
                    "Scenario": scenario_name,
                    GROUP_COL: group_value,
                    "Feature": feat,
                    "Coefficient": round(sm_group_model.params[feat], 4),
                    "p-value": round(sm_group_model.pvalues[feat], 6),
                })
        except Exception as e:
            print(f"Statsmodels failed for {group_value}: {e}")

    importance_df = pd.DataFrame(importance_list)
    coef_group_df = pd.DataFrame(coef_list_by_group)
    pvalues_group_df = pd.DataFrame(pvalues_list_by_group)

    if not importance_df.empty:
        journal_short_map = (
            df[[GROUP_COL, "journal_short"]]
            .drop_duplicates()
            .set_index(GROUP_COL)["journal_short"]
            .to_dict()
        )
        importance_df["journal_full"] = importance_df[GROUP_COL]
        importance_df[GROUP_COL] = (
            importance_df[GROUP_COL]
            .map(journal_short_map)
            .fillna(importance_df[GROUP_COL])
        )
        importance_df = importance_df.set_index(GROUP_COL).sort_values(by="N", ascending=False)

    importance_df.to_csv(
        os.path.join(scenario_dir, f"shap_importance_per_journal_{scenario_name}.csv")
    )
    coef_group_df.to_csv(
        os.path.join(scenario_dir, f"logistic_coefficients_per_journal_{scenario_name}.csv"),
        index=False,
    )
    pvalues_group_df.to_csv(
        os.path.join(scenario_dir, f"logistic_pvalues_per_journal_{scenario_name}.csv"),
        index=False,
    )

    display(importance_df)
    display(coef_group_df)
    display(pvalues_group_df)

    if not importance_df.empty and "N" in importance_df.columns:
        importance_df_plot = importance_df.drop(
            columns=["N", "journal_full"],
            errors="ignore",
        ).copy()

        # Keep the heatmap focused on the same substantive variables used in
        # the main SHAP figures; availability remains a model control only.
        heatmap_features = [
            col for col in PLOT_FOCAL_FEATURES
            if col in importance_df_plot.columns
        ]
        importance_df_plot = importance_df_plot[heatmap_features].copy()
        importance_df_plot = importance_df_plot.rename(
            columns=lambda c: clean_feature_name(c, journal_short_map)
        )

        if not importance_df_plot.empty:
            dynamic_height = max(
                FIGSIZE_HEATMAP[1],
                0.46 * len(importance_df_plot.index) + 1.8,
            )
            fig, ax = plt.subplots(
                figsize=(FIGSIZE_HEATMAP[0], dynamic_height)
            )
            sns.heatmap(
                importance_df_plot,
                annot=True,
                annot_kws={"size": FIG_FONT_SIZE},
                cmap="YlGnBu",
                fmt=".3f",
                linewidths=0.4,
                linecolor="white",
                cbar_kws={"label": "Mean |SHAP value|"},
                ax=ax,
            )
            ax.set_xlabel("")
            ax.set_ylabel("Journal")
            ax.tick_params(axis="x", labelsize=FIG_FONT_SIZE, rotation=30)
            ax.tick_params(axis="y", labelsize=FIG_FONT_SIZE, rotation=0)
            for label in ax.get_xticklabels():
                label.set_horizontalalignment("right")
            apply_figure_fontsize(fig)
            fig.tight_layout()
            save_figure(
                os.path.join(
                    scenario_dir,
                    f"shap_importance_heatmap_per_journal_{scenario_name}",
                )
            )
            plt.show()

    return results_df, pvalues_df

# =========================
# 9b. FULL-SAMPLE INFERENTIAL LOGISTIC REGRESSION
# =========================

def run_inferential_analysis(data, scenario_name, experiment_dict):
    print(f"\n\n########################################")
    print(f"FULL-SAMPLE INFERENCE: {scenario_name}")
    print(f"########################################")

    scenario_dir = os.path.join(OUTPUT_DIR, scenario_name)
    os.makedirs(scenario_dir, exist_ok=True)
    inferential_results = []

    for exp_name, features in experiment_dict.items():
        print(f"\n==============================")
        print(f"INFERENTIAL MODEL: {scenario_name} / {exp_name}")
        print(f"FEATURES: {features}")
        print(f"==============================")

        if not has_required_columns(data, features):
            continue

        model_data = data.dropna(subset=features + ["label"]).copy()
        if len(model_data) < 50 or model_data["label"].nunique() < 2:
            print(f"Skipping {exp_name}: insufficient usable rows/classes")
            continue

        X_infer = build_design_matrix(model_data, features)
        y = model_data["label"].copy()
        X_infer, zero_var = remove_zero_variance_features(X_infer)

        if zero_var:
            print(f"Dropped zero-variance features: {zero_var}")
        if X_infer.shape[1] == 0:
            continue

        # Center year only for numerical stability, keep other units natural.
        if "year" in X_infer.columns:
            X_infer["year"] = X_infer["year"] - X_infer["year"].mean()

        X_sm = sm.add_constant(X_infer, has_constant="add")

        try:
            sm_model = sm.Logit(y, X_sm).fit(disp=0)
            conf = sm_model.conf_int()

            for feat in sm_model.params.index:
                beta = float(sm_model.params[feat])
                ci_low_beta = float(conf.loc[feat, 0])
                ci_high_beta = float(conf.loc[feat, 1])

                row = {
                    "Scenario": scenario_name,
                    "Model": exp_name,
                    "Feature": feat,
                    "N": len(model_data),
                    "Coefficient": beta,
                    "Std_Error": float(sm_model.bse[feat]),
                    "Odds_Ratio": float(np.exp(beta)),
                    "OR_CI95_Lower": float(np.exp(ci_low_beta)),
                    "OR_CI95_Upper": float(np.exp(ci_high_beta)),
                    "p-value": float(sm_model.pvalues[feat]),
                    "Pseudo_R2": float(sm_model.prsquared),
                    "AIC": float(sm_model.aic),
                    "BIC": float(sm_model.bic),
                }

                if feat == "less_frequent_citation_ratio":
                    row["OR_per_0.10_ratio_increase"] = float(np.exp(beta * 0.10))
                    row["OR_per_0.10_CI95_Lower"] = float(np.exp(ci_low_beta * 0.10))
                    row["OR_per_0.10_CI95_Upper"] = float(np.exp(ci_high_beta * 0.10))

                inferential_results.append(row)

        except Exception as e:
            print(f"Inferential statsmodels failed for {scenario_name} / {exp_name}: {e}")

    inferential_df = pd.DataFrame(inferential_results)
    inferential_path = os.path.join(
        scenario_dir,
        f"logistic_inferential_fullsample_{scenario_name}.csv",
    )
    inferential_df.to_csv(inferential_path, index=False)
    print("\nSaved full-sample inferential results:")
    print(inferential_path)
    display(inferential_df)
    return inferential_df


def run_journal_moderation_analysis(data, scenario_name, main_features):
    """Test whether the LFCR -> article-impact association differs by journal."""
    print(f"\n\n########################################")
    print(f"JOURNAL MODERATION TEST: {scenario_name}")
    print(f"########################################")

    scenario_dir = os.path.join(OUTPUT_DIR, scenario_name)
    os.makedirs(scenario_dir, exist_ok=True)

    model_data = data.dropna(subset=main_features + ["label"]).copy()
    if len(model_data) < 50 or model_data["label"].nunique() < 2:
        print("Moderation skipped: insufficient data.")
        return pd.DataFrame(), pd.DataFrame()

    observed_journals = sorted(model_data[GROUP_COL].astype(str).unique())
    reference_journal = observed_journals[0]

    X_reduced = build_design_matrix(model_data, main_features)
    y = model_data["label"].copy()
    X_reduced, _ = remove_zero_variance_features(X_reduced)

    if "year" in X_reduced.columns:
        X_reduced["year"] = X_reduced["year"] - X_reduced["year"].mean()

    journal_dummy_cols = [
        c for c in X_reduced.columns if c.startswith(f"{GROUP_COL}_")
    ]

    X_full = X_reduced.copy()
    for dummy_col in journal_dummy_cols:
        interaction_col = f"less_frequent_citation_ratio_x_{dummy_col}"
        X_full[interaction_col] = (
            X_full["less_frequent_citation_ratio"] * X_full[dummy_col]
        )

    X_full, _ = remove_zero_variance_features(X_full)

    try:
        reduced_model = sm.Logit(
            y, sm.add_constant(X_reduced, has_constant="add")
        ).fit(disp=0)
        full_model = sm.Logit(
            y, sm.add_constant(X_full, has_constant="add")
        ).fit(disp=0)

        lr_stat = 2.0 * (full_model.llf - reduced_model.llf)
        df_diff = int(full_model.df_model - reduced_model.df_model)
        lr_pvalue = float(chi2.sf(lr_stat, df_diff)) if df_diff > 0 else np.nan

        joint_test_df = pd.DataFrame([{
            "Scenario": scenario_name,
            "Reference_Journal": reference_journal,
            "N": len(model_data),
            "LR_statistic": float(lr_stat),
            "df": df_diff,
            "p-value": lr_pvalue,
            "Interpretation": "Significant => LFCR-impact association differs across journals",
        }])

        conf = full_model.conf_int()
        interaction_rows = []
        for term in full_model.params.index:
            if not term.startswith("less_frequent_citation_ratio_x_journal_"):
                continue
            beta = float(full_model.params[term])
            ci_low = float(conf.loc[term, 0])
            ci_high = float(conf.loc[term, 1])
            interaction_rows.append({
                "Scenario": scenario_name,
                "Reference_Journal": reference_journal,
                "Interaction_Term": term,
                "Coefficient": beta,
                "Std_Error": float(full_model.bse[term]),
                "p-value": float(full_model.pvalues[term]),
                "Differential_OR_per_0.10_ratio_increase": float(np.exp(beta * 0.10)),
                "Differential_OR_per_0.10_CI95_Lower": float(np.exp(ci_low * 0.10)),
                "Differential_OR_per_0.10_CI95_Upper": float(np.exp(ci_high * 0.10)),
            })

        interactions_df = pd.DataFrame(interaction_rows)

        # ---------------------------------------------------------
        # Multiple-comparison correction for journal-specific
        # post-hoc interaction tests.
        #
        # IMPORTANT:
        # - The joint LR chi-square test above is a single omnibus test,
        #   so its p-value is NOT adjusted.
        # - BH/FDR is applied only to the family of individual
        #   LFCR x journal interaction p-values reported below.
        # - Journal-specific post-hoc findings should be interpreted
        #   primarily when the joint LR test indicates overall moderation.
        # ---------------------------------------------------------
        if not interactions_df.empty:
            raw_pvalues = interactions_df["p-value"].astype(float).to_numpy()

            reject_bh, pvalues_bh, _, _ = multipletests(
                raw_pvalues,
                alpha=0.05,
                method="fdr_bh",
            )

            interactions_df["p_BH"] = pvalues_bh
            interactions_df["significant_raw_0.05"] = raw_pvalues < 0.05
            interactions_df["significant_BH_0.05"] = reject_bh
            interactions_df["BH_method"] = "Benjamini-Hochberg FDR"
            interactions_df["joint_LR_pvalue"] = lr_pvalue
            interactions_df["joint_LR_significant_0.05"] = (
                bool(lr_pvalue < 0.05) if np.isfinite(lr_pvalue) else False
            )
        else:
            # Keep a stable output schema even if no interaction terms exist.
            interactions_df["p_BH"] = pd.Series(dtype=float)
            interactions_df["significant_raw_0.05"] = pd.Series(dtype=bool)
            interactions_df["significant_BH_0.05"] = pd.Series(dtype=bool)
            interactions_df["BH_method"] = pd.Series(dtype=str)
            interactions_df["joint_LR_pvalue"] = pd.Series(dtype=float)
            interactions_df["joint_LR_significant_0.05"] = pd.Series(dtype=bool)

        joint_test_df.to_csv(
            os.path.join(
                scenario_dir,
                f"journal_moderation_joint_LR_test_{scenario_name}.csv",
            ),
            index=False,
        )
        interactions_df.to_csv(
            os.path.join(
                scenario_dir,
                f"journal_moderation_interactions_{scenario_name}.csv",
            ),
            index=False,
        )

        print("Reference journal:", reference_journal)
        display(joint_test_df)
        display(interactions_df)
        return joint_test_df, interactions_df

    except Exception as e:
        print(f"Journal moderation analysis failed for {scenario_name}: {e}")
        return pd.DataFrame(), pd.DataFrame()

# =========================
# 10. RUN BOTH SCENARIOS
# =========================

In [ ]:
# =========================
# Predictive ANALYSES
# =========================

results_valid, pvalues_valid = run_analysis(
    data=df_valid,
    scenario_name="valid_only",
    experiment_dict=experiment_dict_valid,
    final_model_name="M4_ratio_adjusted"
)

results_full, pvalues_full = run_analysis(
    data=df_full,
    scenario_name="full_dataset",
    experiment_dict=experiment_dict_full,
    final_model_name="M4_ratio_adjusted"
)

# =========================
# FULL-SAMPLE INFERENTIAL ANALYSES
# =========================

inferential_valid = run_inferential_analysis(
    data=df_valid,
    scenario_name="valid_only",
    experiment_dict=experiment_dict_valid
)

inferential_full = run_inferential_analysis(
    data=df_full,
    scenario_name="full_dataset",
    experiment_dict=experiment_dict_full
)

# =========================
# FORMAL JOURNAL MODERATION TESTS
# =========================
moderation_joint_valid, moderation_interactions_valid = run_journal_moderation_analysis(
    data=df_valid,
    scenario_name="valid_only",
    main_features=experiment_dict_valid["M4_ratio_adjusted"],
)

moderation_joint_full, moderation_interactions_full = run_journal_moderation_analysis(
    data=df_full,
    scenario_name="full_dataset",
    main_features=experiment_dict_full["M4_ratio_adjusted"],
)

## 10b. Additional LFC analyses (M5) and matched M3 comparisons

Original M4 analyses above are unchanged. The following **additional** outputs assess whether LFC presence carries an adjusted early-impact association. Nested LR models are fitted on exactly the **same** rows and in the **same natural coefficient units** (year centered), with journal fixed effects. BH is shown as a supplementary multiplicity sensitivity check for the two alternative focal tests within each analysis scenario; it was added during final reporting and is not presented as preregistered. Descriptive rates exclude unobserved LFC from the observed no-LFC category.

In [ ]:
# =========================
# 10b. LFC-SPECIFIC ANALYSIS HELPERS
# =========================
from statsmodels.stats.proportion import proportion_confint

LFC = "has_less_frequent_citation"
LFCR = "less_frequent_citation_ratio"
ALTERNATIVE_MODEL_NAMES = ["M4_ratio_adjusted", "M5_presence_adjusted"]


def fit_natural_logit_on_common_sample(y, raw_X):
    """Journal dummies 0/1, natural LFCR/references, centered year; no standardizing."""
    X_fit = raw_X.copy().astype(float)
    if "year" in X_fit.columns:
        X_fit["year"] -= X_fit["year"].mean()
    return safe_logit_fit(y, X_fit)


def matched_incremental_lr(data, scenario_name, experiments):
    """M3→M4 (LFCR) and M3→M5 (LFC): the same rows in all three fits."""
    control_key = ("M3_controls_only" if "M3_controls_only" in experiments
                   else "M3_controls_plus_availability")
    controls = experiments[control_key]
    union = list(dict.fromkeys(controls + [LFCR, LFC]))
    subset = data.dropna(subset=union + ["label"]).copy()
    empty_cols = ["Scenario", "Comparison", "Added_feature", "N", "LR_statistic",
                  "df", "p-value", "AIC_M3", "AIC_augmented", "p_BH",
                  "significant_raw_0.05", "significant_BH_0.05"]
    if len(subset) < 50 or subset["label"].nunique() < 2:
        print(f"No matched-sample incremental LR for {scenario_name}: insufficient sample")
        return pd.DataFrame(columns=empty_cols)

    X_controls = build_design_matrix(subset, controls)
    X_controls, _ = remove_zero_variance_features(X_controls)
    y = subset["label"]
    rows = []
    try:
        # Centering year identically within the shared sample preserves nesting.
        if "year" in X_controls:
            X_controls["year"] = X_controls["year"] - X_controls["year"].mean()
        reduced = safe_logit_fit(y, X_controls)
        for model_name, focal in [("M4_ratio_adjusted", LFCR), ("M5_presence_adjusted", LFC)]:
            X_full = X_controls.copy()
            if focal in X_full:
                raise ValueError(f"Focal predictor {focal} already present in controls")
            X_full[focal] = pd.to_numeric(subset[focal], errors="raise").astype(float)
            if X_full[focal].nunique() < 2:
                print(f"No LR for {model_name}: {focal} is constant in this scenario")
                rows.append({"Scenario": scenario_name, "Comparison": f"M3_vs_{model_name}",
                             "Added_feature": focal, "N": len(subset), "LR_statistic": np.nan,
                             "df": 0, "p-value": np.nan, "AIC_M3": reduced.aic, "AIC_augmented": np.nan})
                continue
            augmented = safe_logit_fit(y, X_full)
            LR = max(0.0, float(2 * (augmented.llf - reduced.llf)))
            df_diff = int(augmented.df_model - reduced.df_model)
            rows.append({
                "Scenario": scenario_name,
                "Comparison": f"M3_vs_{model_name}",
                "Added_feature": focal,
                "N": int(len(subset)),
                "LR_statistic": LR,
                "df": df_diff,
                "p-value": float(chi2.sf(LR, df_diff)) if df_diff > 0 else np.nan,
                "AIC_M3": float(reduced.aic),
                "AIC_augmented": float(augmented.aic),
            })
    except Exception as e:
        print(f"Matched incremental LR unavailable for {scenario_name}: {e}")
        return pd.DataFrame(columns=empty_cols)

    result = pd.DataFrame(rows)
    result["p_BH"] = np.nan
    result["significant_raw_0.05"] = False
    result["significant_BH_0.05"] = False
    good = result["p-value"].notna()
    if good.any():
        reject, adj_p, _, _ = multipletests(
            result.loc[good, "p-value"].astype(float).to_numpy(),
            alpha=0.05, method="fdr_bh"
        )
        result.loc[good, "p_BH"] = adj_p
        result.loc[good, "significant_raw_0.05"] = result.loc[good, "p-value"] < 0.05
        result.loc[good, "significant_BH_0.05"] = reject
    target = os.path.join(OUTPUT_DIR, scenario_name, f"nested_LR_M3_vs_M4_M5_{scenario_name}.csv")
    result.to_csv(target, index=False)
    display(result)
    return result


def focal_fdr_table(inferential_valid, inferential_full):
    """BH sensitivity check for the two alternative adjusted feature tests, per scenario."""
    rows = []
    for scenario, inf in [("valid_only", inferential_valid), ("full_dataset", inferential_full)]:
        for model, feature in [("M4_ratio_adjusted", LFCR), ("M5_presence_adjusted", LFC)]:
            hit = inf.loc[(inf["Model"] == model) & (inf["Feature"] == feature)]
            if len(hit):
                z = hit.iloc[0]
                is_ratio = (feature == LFCR)
                # Report LFCR on the scientifically used +0.10 scale; report
                # binary LFC as a 1-versus-0 odds ratio (not a standardized beta).
                or_col = "OR_per_0.10_ratio_increase" if is_ratio else "Odds_Ratio"
                lo_col = "OR_per_0.10_CI95_Lower" if is_ratio else "OR_CI95_Lower"
                hi_col = "OR_per_0.10_CI95_Upper" if is_ratio else "OR_CI95_Upper"
                rows.append({"Scenario": scenario, "Model": model, "Feature": feature,
                             "Effect_Contrast": "+0.10 LFCR" if is_ratio else "LFC=1 vs LFC=0",
                             "N": int(z["N"]), "OR": float(z[or_col]),
                             "OR_CI95_Lower": float(z[lo_col]),
                             "OR_CI95_Upper": float(z[hi_col]),
                             "p-value": float(z["p-value"])})
    df_focal = pd.DataFrame(rows)
    if df_focal.empty:
        return df_focal
    df_focal["p_BH"] = np.nan
    df_focal["significant_BH_0.05"] = False
    for scenario, ids in df_focal.groupby("Scenario").groups.items():
        reject, p_adj, _, _ = multipletests(df_focal.loc[ids, "p-value"], method="fdr_bh", alpha=0.05)
        df_focal.loc[ids, "p_BH"] = p_adj
        df_focal.loc[ids, "significant_BH_0.05"] = reject
    df_focal.to_csv(os.path.join(OUTPUT_DIR, "adjusted_LFCR_LFC_focal_BH.csv"), index=False)
    display(df_focal)
    return df_focal


def make_binary_lfc_rates(data, by_journal=False):
    """Observed LFC=0/1 only: call with valid_only, NEVER imputed full rows."""
    if not data["has_valid_citation_data"].eq(1).all():
        raise ValueError("Observed LFC proportions require genuinely valid citation data only")
    groups = (["journal", "journal_short"] if by_journal else []) + [LFC]
    grouped = (data.groupby(groups, dropna=False)["label"]
               .agg(N="size", High_N="sum").reset_index())
    grouped["Low_N"] = grouped["N"] - grouped["High_N"]
    grouped["High_rate"] = grouped["High_N"] / grouped["N"]
    cis = [proportion_confint(k, n, alpha=0.05, method="wilson")
           for k,n in zip(grouped["High_N"], grouped["N"])]
    grouped["High_rate_CI95_Lower"] = [c[0] for c in cis]
    grouped["High_rate_CI95_Upper"] = [c[1] for c in cis]
    grouped["Observation"] = "valid_only_observed_LFC"
    return grouped


def save_lfc_descriptives(valid_data, full_data):
    all_rates = make_binary_lfc_rates(valid_data)
    journal_rates = make_binary_lfc_rates(valid_data, by_journal=True)
    all_rates.to_csv(os.path.join(OUTPUT_DIR, "lfc_observed_high_impact_rates_valid_only.csv"), index=False)
    journal_rates.to_csv(os.path.join(OUTPUT_DIR, "lfc_observed_high_impact_by_journal_valid_only.csv"), index=False)
    display(all_rates)
    display(journal_rates)

    # The unavailable group is separate, not assumed to be observed LFC=0.
    descriptive = full_data.copy()
    descriptive["LFC_group"] = np.select(
        [descriptive["has_valid_citation_data"].eq(0), descriptive[LFC].eq(1)],
        ["Citation data unavailable", "Observed LFC = 1"],
        default="Observed LFC = 0"
    )
    grouped = (descriptive.groupby("LFC_group")["label"]
               .agg(N="size", High_N="sum", High_rate="mean").reset_index())
    grouped.to_csv(os.path.join(OUTPUT_DIR, "lfc_full_dataset_availability_strata.csv"), index=False)

    # Main plot ONLY shows genuinely observed LFC=0 and LFC=1.
    plot = all_rates.sort_values(LFC)
    if len(plot) < 2:
        print("Observed LFC plot skipped: only one LFC group is available")
        return all_rates, journal_rates, grouped
    fig, ax = plt.subplots(figsize=FIGSIZE_MAIN)
    bars = ax.bar(["LFC = 0", "LFC = 1"], plot["High_rate"].to_numpy(), width=0.22, color=BAR_COLOR)
    err_low = plot["High_rate"].to_numpy() - plot["High_rate_CI95_Lower"].to_numpy()
    err_high = plot["High_rate_CI95_Upper"].to_numpy() - plot["High_rate"].to_numpy()
    ax.errorbar([0, 1], plot["High_rate"], yerr=[err_low, err_high], fmt="none", capsize=4)
    for bar, n, high, ci_upper in zip(bars, plot["N"], plot["High_rate"], plot["High_rate_CI95_Upper"]):
        ax.text(bar.get_x() + bar.get_width()/2, ci_upper + 0.02,
                f"{high:.1%}\n(n={int(n)})", ha="center", va="bottom")
    ax.set_ylim(0, max(0.75, min(1.0, float(plot["High_rate_CI95_Upper"].max()+0.16))))
    ax.set_ylabel("Observed proportion of high-impact papers")
    ax.set_xlabel("LFC presence (valid papers only)")
    ax.grid(axis="x", visible=False)
    ax.grid(axis="y", alpha=0.25)
    apply_figure_fontsize(fig)
    fig.tight_layout()
    save_figure(os.path.join(OUTPUT_DIR, "lfc_observed_high_impact_valid_only"))
    plt.show()
    return all_rates, journal_rates, grouped


def make_lfc_adjusted_effect_table(inferential_valid, inferential_full):
    rows = []
    for scenario, inf in [("valid_only", inferential_valid), ("full_dataset", inferential_full)]:
        hit = inf.loc[(inf["Model"] == "M5_presence_adjusted") & (inf["Feature"] == LFC)]
        if not len(hit):
            print(f"No inferential LFC row found for {scenario}")
            continue
        r = hit.iloc[0]
        rows.append({"Scenario": scenario, "Model": "M5_presence_adjusted", "Feature": LFC,
                     "N": int(r["N"]), "Coefficient": float(r["Coefficient"]),
                     "OR_LFC1_vs_0": float(r["Odds_Ratio"]),
                     "CI95_Lower": float(r["OR_CI95_Lower"]),
                     "CI95_Upper": float(r["OR_CI95_Upper"]), "p-value": float(r["p-value"])})
    effects = pd.DataFrame(rows)
    effects.to_csv(os.path.join(OUTPUT_DIR, "lfc_adjusted_M5_OR_CI95_valid_full.csv"), index=False)
    if effects.empty: return effects
    fig, ax = plt.subplots(figsize=(6, 2.5))
    ys = np.arange(len(effects))[::-1]
    x = effects["OR_LFC1_vs_0"].to_numpy()
    lo = effects["CI95_Lower"].to_numpy()
    hi = effects["CI95_Upper"].to_numpy()
    ax.errorbar(x, ys, xerr=np.vstack((x-lo, hi-x)), fmt="o", capsize=4)
    ax.axvline(1.0, linestyle="--", linewidth=1)
    ax.set_yticks(ys)
    ax.set_yticklabels(["Valid-only" if s == "valid_only" else "Full dataset" for s in effects["Scenario"]])
    ax.set_xscale("log")
    ax.set_xlim(
        max(0.1, min(0.9, float(lo.min() * 0.80))),
        max(1.1, float(hi.max() * 1.20))
    )
    # Ευανάγνωστες ετικέτες αντί για 10^0, 2×10^0
    ticks = [
        v for v in [0.5, 0.75, 1, 1.5, 2, 3]
        if ax.get_xlim()[0] <= v <= ax.get_xlim()[1]
    ]
    ax.set_xticks(ticks)
    ax.set_xticklabels([f"{v:g}" for v in ticks])
    # Απενεργοποίηση των αυτόματων minor ticks
    ax.minorticks_off()
    ax.set_xlabel("Adjusted odds ratio (LFC = 1 vs 0), 95% CI")
    ax.set_ylabel("")
    ax.grid(axis="x", alpha=0.25)
    ax.grid(axis="y", visible=False)
    apply_figure_fontsize(fig)
    fig.tight_layout()
    save_figure(os.path.join(OUTPUT_DIR, "lfc_adjusted_M5_forest_valid_full"))
    plt.show()
    display(effects)
    return effects


def run_m5_shap(data, scenario_name, experiments):
    """Fit full M5 with all controls, visually expose LFC/References/Year only."""
    folder = os.path.join(OUTPUT_DIR, scenario_name)
    features = experiments["M5_presence_adjusted"]
    subset = data.dropna(subset=features+["label"]).copy()
    if len(subset) < 50 or subset["label"].nunique() < 2:
        print(f"M5 SHAP skipped for {scenario_name}: insufficient rows/classes")
        return pd.DataFrame()
    X = build_design_matrix(subset, features)
    X, zero_var = remove_zero_variance_features(X)
    Xs = scale_predictive_full(X)
    if LFC not in Xs.columns or Xs[LFC].nunique() < 2:
        print(f"M5 SHAP skipped for {scenario_name}: LFC has no variance")
        return pd.DataFrame()
    model = LogisticRegression(max_iter=1000)
    model.fit(Xs, subset["label"])
    explainer = shap.Explainer(model, Xs, feature_names=Xs.columns.tolist())
    shap_values = explainer(Xs)
    full_importance = pd.DataFrame({
        "Feature": Xs.columns.tolist(),
        "MeanAbsSHAP": np.abs(shap_values.values).mean(axis=0),
        "N": len(subset), "Model": "M5_presence_adjusted", "Scenario": scenario_name
    }).sort_values("MeanAbsSHAP", ascending=False)
    full_importance.to_csv(os.path.join(folder, f"shap_global_importance_M5_presence_adjusted_{scenario_name}.csv"), index=False)
    main_cols = [col for col in PLOT_LFC_FEATURES if col in Xs.columns]
    indices = [Xs.columns.get_loc(c) for c in main_cols]
    plot_X = Xs[main_cols].copy()
    short_names = [clean_feature_name(c, get_journal_short_map(data)) for c in main_cols]
    shap.summary_plot(shap_values.values[:, indices], plot_X,
                      feature_names=short_names, show=False,
                      plot_size=FIGSIZE_WIDE, max_display=len(main_cols))
    fig = plt.gcf()
    ax = plt.gca()
    ax.set_xlim(-0.6, 3.8)
    ax.set_xticks([-0.5, 0, 0.5, 1, 1.5, 2, 2.5, 3, 3.5])
    ax.set_xlabel("SHAP value (M5)")
    apply_figure_fontsize(fig)
    fig.tight_layout()
    save_figure(os.path.join(folder, f"shap_summary_global_M5_presence_adjusted_{scenario_name}"))
    plt.show()

    bar = full_importance.loc[full_importance["Feature"].isin(main_cols)].copy()
    bar["Feature_Label"] = bar["Feature"].apply(clean_feature_name)
    fig, ax = plt.subplots(figsize=FIGSIZE_MAIN)

    #sns.barplot(bar["Feature_Label"], bar["MeanAbsSHAP"], color=BAR_COLOR)
    #ax.invert_yaxis()

    sns.barplot(
        data=bar,
        x="MeanAbsSHAP",
        y="Feature_Label",
        ax=ax,
        color=BAR_COLOR,
        saturation=1,
        width=0.7,
    )

    ax.set_xlabel("Mean |SHAP value| (M5)")
    ax.set_xlim(0, 0.50)
    ax.set_xticks(np.arange(0, 0.501, 0.05))
    ax.set_ylabel("")
    ax.grid(axis="x", alpha=0.25)
    ax.grid(axis="y", visible=False)
    apply_figure_fontsize(fig)
    fig.tight_layout()
    save_figure(os.path.join(folder, f"shap_global_barplot_M5_presence_adjusted_{scenario_name}"))
    plt.show()
    return full_importance


def make_predictive_M3_M4_M5_summary(results_valid, results_full):
    combo = pd.concat([results_valid, results_full], ignore_index=True)
    combo = combo.loc[combo["Model"].isin([
        "M3_controls_only", "M3_controls_plus_availability",
        "M4_ratio_adjusted", "M5_presence_adjusted"
    ])].copy()
    baseline = combo.loc[combo["Model"].isin([
        "M3_controls_only", "M3_controls_plus_availability"
    ]), ["Scenario", "ROC_AUC"]]
    baseline = baseline.rename(columns={"ROC_AUC": "M3_ROC_AUC_baseline"})
    combo = combo.merge(baseline, on="Scenario", how="left")
    combo["Delta_AUC_vs_M3"] = combo["ROC_AUC"] - combo["M3_ROC_AUC_baseline"]
    combo["Interpretation"] = "Single held-out split; delta is descriptive (no significance test)"
    combo.to_csv(os.path.join(OUTPUT_DIR, "predictive_comparison_M3_M4_M5_valid_full.csv"), index=False)
    display(combo)
    return combo


## 10c. Run extra LFCR/LFC tests and produce LFC figures

These are **additions**, not replacements of existing M4 output files. Inference and exploratory predictive discrimination are reported separately. The figure comparing observed High-impact rates is restricted to observed (valid-only) LFC status; missing citation-function data remain a separate category in the full descriptive CSV.

In [ ]:
# =========================
# 10c. COMPLETE LFC SUPPORT: RUN
# =========================

# Formal paired incremental comparisons using natural, full-sample coefficients.
nested_valid = matched_incremental_lr(df_valid, "valid_only", experiment_dict_valid)
nested_full = matched_incremental_lr(df_full, "full_dataset", experiment_dict_full)
nested_both = pd.concat([nested_valid, nested_full], ignore_index=True)
nested_both.to_csv(os.path.join(OUTPUT_DIR, "nested_LR_M3_vs_M4_M5_VALID_AND_FULL.csv"), index=False)

# Supplementary multiplicity sensitivity for the two focal M4/M5 adjusted coefficient tests.
focal_bh = focal_fdr_table(inferential_valid, inferential_full)

# Observed presence vs absence: valid rows only (no false zero-label for missing data).
lfc_rates, lfc_journal_rates, lfc_availability_strata = save_lfc_descriptives(df_valid, df_full)

# Adjusted OR and 95% CI for M5 binary LFC presence; 11-pt PNG/PDF forest plot.
lfc_effects = make_lfc_adjusted_effect_table(inferential_valid, inferential_full)

# SHAP for the fully adjusted M5 models in parallel with original M4 SHAP.
m5_shap_valid = run_m5_shap(df_valid, "valid_only", experiment_dict_valid)
m5_shap_full = run_m5_shap(df_full, "full_dataset", experiment_dict_full)

# The two alternative operationalizations, alongside the identical controls-only baseline.
comparison_predictive_M3_M4_M5 = make_predictive_M3_M4_M5_summary(results_valid, results_full)


# =========================
# 11. SAVE COMBINED RESULTS
# =========================

In [ ]:
combined_performance = pd.concat([results_valid, results_full], ignore_index=True)
combined_pvalues = pd.concat([pvalues_valid, pvalues_full], ignore_index=True)

combined_performance.to_csv(
    os.path.join(OUTPUT_DIR, "logistic_performance_results_VALID_AND_FULL.csv"),
    index=False
)

combined_pvalues.to_csv(
    os.path.join(OUTPUT_DIR, "logistic_pvalues_results_VALID_AND_FULL.csv"),
    index=False
)

print("Combined outputs saved to:", OUTPUT_DIR)
display(combined_performance)
display(combined_pvalues)

combined_inferential = pd.concat(
    [inferential_valid, inferential_full],
    ignore_index=True
)

combined_inferential_path = os.path.join(
    OUTPUT_DIR,
    "logistic_inferential_results_VALID_AND_FULL.csv"
)

combined_inferential.to_csv(
    combined_inferential_path,
    index=False
)

print(
    "Combined full-sample inferential results saved to:"
)
print(combined_inferential_path)

display(combined_inferential)

# =========================
# 12. AUTOMATIC MILESTONE COMPARISON
# =========================

In [ ]:
# =========================
# 12. CROSS-MILESTONE COMPARISON — ORIGINAL FIELDS + LFC EXTENSION
# =========================
# One reconstructed row per milestone, never append duplicates.
# Existing mil1/mil2/mil3 values remain readable; new LR fields will appear
# for a milestone only after that milestone has been rerun using this version.

comparison_rows = []

for milestone in sorted(os.listdir(MILESTONES_BASE_DIR)):
    milestone_dir = os.path.join(MILESTONES_BASE_DIR, milestone)
    metadata_path = os.path.join(milestone_dir, "metadata.csv")
    pvalues_path = os.path.join(milestone_dir, "outputs", "logistic_inferential_results_VALID_AND_FULL.csv")
    performance_path = os.path.join(milestone_dir, "outputs", "logistic_performance_results_VALID_AND_FULL.csv")
    moderation_path = os.path.join(milestone_dir, "outputs", "full_dataset",
                                   "journal_moderation_joint_LR_test_full_dataset.csv")
    nested_path = os.path.join(milestone_dir, "outputs", "nested_LR_M3_vs_M4_M5_VALID_AND_FULL.csv")
    bh_path = os.path.join(milestone_dir, "outputs", "adjusted_LFCR_LFC_focal_BH.csv")
    observed_path = os.path.join(milestone_dir, "outputs", "lfc_observed_high_impact_rates_valid_only.csv")

    if not os.path.isfile(metadata_path):
        continue
    metadata = pd.read_csv(metadata_path, index_col=0, header=None).squeeze("columns").to_dict()
    row = {
        "milestone": milestone,
        "milestone_filtering": metadata.get("milestone_filtering"),
        "current_milestone_order": metadata.get("current_milestone_order"),
        "included_add_in_milestones": metadata.get("included_add_in_milestones"),
        "n_total_low_high": metadata.get("n_total_low_high"),
        "n_valid_low_high": metadata.get("n_valid_low_high"),
        "coverage": metadata.get("coverage"),
        "n_journals": metadata.get("n_journals"),
    }

    if os.path.isfile(pvalues_path):
        pv = pd.read_csv(pvalues_path)
        targets = [
            ("valid_only", "M1_ratio_only"),
            ("valid_only", "M2_presence_only"),
            ("valid_only", "M4_ratio_adjusted"),
            ("valid_only", "M5_presence_adjusted"),
            ("full_dataset", "M1_ratio_plus_availability"),
            ("full_dataset", "M2_presence_plus_availability"),
            ("full_dataset", "M4_ratio_adjusted"),
            ("full_dataset", "M5_presence_adjusted"),
        ]
        target_features = [LFCR, LFC, "has_valid_citation_data", "references", "year"]
        for scenario_name, model_name in targets:
            subset = pv.loc[(pv["Scenario"] == scenario_name) & (pv["Model"] == model_name)]
            for feature in target_features:
                hit = subset.loc[subset["Feature"] == feature]
                if hit.empty: continue
                z = hit.iloc[0]
                prefix = f"{scenario_name}_{model_name}_{feature}"
                row[f"{prefix}_coef"] = z.get("Coefficient")
                row[f"{prefix}_pvalue"] = z.get("p-value")
                # Expanded main manuscript fields: natural-unit M5 OR and +0.10 M4 OR.
                if feature in (LFCR, LFC):
                    row[f"{prefix}_OR"] = z.get("Odds_Ratio")
                    row[f"{prefix}_CI95_Lower"] = z.get("OR_CI95_Lower")
                    row[f"{prefix}_CI95_Upper"] = z.get("OR_CI95_Upper")
                    if feature == LFCR:
                        for key in ["OR_per_0.10_ratio_increase", "OR_per_0.10_CI95_Lower", "OR_per_0.10_CI95_Upper"]:
                            row[f"{prefix}_{key}"] = z.get(key)
                if feature == LFC:
                    row[f"{prefix}_AIC"] = z.get("AIC")

    if os.path.isfile(performance_path):
        perf = pd.read_csv(performance_path)
        for scenario in ("valid_only", "full_dataset"):
            ctrl = "M3_controls_only" if scenario == "valid_only" else "M3_controls_plus_availability"
            for model, short in [(ctrl, "M3"), ("M4_ratio_adjusted", "M4"), ("M5_presence_adjusted", "M5")]:
                hit = perf.loc[(perf["Scenario"] == scenario) & (perf["Model"] == model)]
                if hit.empty: continue
                z = hit.iloc[0]
                for stat in ("Accuracy", "F1", "Precision", "Recall", "ROC_AUC"):
                    row[f"{scenario}_{short}_{stat.lower()}"] = z.get(stat)

    if os.path.isfile(moderation_path):
        mod = pd.read_csv(moderation_path)
        if not mod.empty:
            row["full_dataset_journal_moderation_LR"] = mod["LR_statistic"].iloc[0]
            row["full_dataset_journal_moderation_df"] = mod["df"].iloc[0]
            row["full_dataset_journal_moderation_pvalue"] = mod["p-value"].iloc[0]

    # Newly calculated matched-sample M3→M4 and M3→M5 comparison, BH-adjusted.
    if os.path.isfile(nested_path):
        nested = pd.read_csv(nested_path)
        for _, z in nested.iterrows():
            short = "M4" if z["Added_feature"] == LFCR else "M5"
            prefix = f'{z["Scenario"]}_M3_vs_{short}'
            for out_name, in_name in [("N", "N"), ("LR", "LR_statistic"),
                                      ("df", "df"), ("pvalue", "p-value"),
                                      ("p_BH", "p_BH"), ("AIC_M3", "AIC_M3"),
                                      ("AIC_augmented", "AIC_augmented")]:
                row[f"{prefix}_{out_name}"] = z.get(in_name)

    if os.path.isfile(bh_path):
        bh = pd.read_csv(bh_path)
        for _, z in bh.iterrows():
            short = "M4" if z["Feature"] == LFCR else "M5"
            row[f'{z["Scenario"]}_{short}_focal_p_BH'] = z.get("p_BH")

    if os.path.isfile(observed_path):
        obs = pd.read_csv(observed_path)
        for _, z in obs.iterrows():
            row[f'valid_only_LFC{int(z[LFC])}_observed_N'] = z.get("N")
            row[f'valid_only_LFC{int(z[LFC])}_observed_High_rate'] = z.get("High_rate")

    comparison_rows.append(row)

milestone_comparison_df = pd.DataFrame(comparison_rows)
if not milestone_comparison_df.empty:
    milestone_comparison_df = (milestone_comparison_df
                               .sort_values("current_milestone_order", kind="stable")
                               .reset_index(drop=True))
comparison_out = os.path.join(MILESTONES_BASE_DIR, "journal_milestone_comparison_summary.csv")
# One-time backup of the previous comparative view before updating its schema.
backup = os.path.join(MILESTONES_BASE_DIR, "journal_milestone_comparison_summary_pre_LFC.csv")
if os.path.isfile(comparison_out) and not os.path.isfile(backup):
    shutil.copy2(comparison_out, backup)
milestone_comparison_df.to_csv(comparison_out, index=False)
print("One row per milestone; previous versions remain in milestone-specific outputs.")
print("Comparison saved:", comparison_out)
display(milestone_comparison_df)

# Verify immutable input files still have the exact same bytes after the rerun.
source_hashes_after = {name: input_sha256(path) for name, path in source_paths.items()}
if source_hashes_before != source_hashes_after:
    raise RuntimeError("A source input was unexpectedly changed during the analysis")
manifest = pd.DataFrame({
    "Source_file": list(source_paths.keys()),
    "Path_used": list(source_paths.values()),
    "SHA256_before": list(source_hashes_before.values()),
    "SHA256_after": list(source_hashes_after.values()),
    "Snapshot_mode": SNAPSHOT_MODE,
})
manifest.to_csv(os.path.join(OUTPUT_DIR, "source_snapshot_integrity_manifest.csv"), index=False)
print("Frozen input integrity: PASS; central rare summary not used in resume mode.")
display(manifest)


### How to rerun frozen milestones safely

1. Keep the existing `mil1`–`mil4` milestone folders, including their `input_files` source snapshots.
2. Mount Drive, run the notebook top-to-bottom and select the milestone as prompted (`v4_coverage_100pct` for mil4).
3. Confirm the console prints `Resume mode: loading from existing milestone snapshot` and `Input mode: FROZEN MILESTONE SNAPSHOT` **before** continuing.
4. Updated M4 and new M5 outputs go to that milestone's `outputs` directory. The consolidated CSV is rebuilt as one row per milestone; if an older milestone has not yet been rerun with this version, newly added fields that require rerunning may be blank for that row.
5. The last cell checks SHA-256 before and after to verify that the three source snapshot CSVs were not modified.

**Important:** `M4` is the initially selected continuous specification. A significant `M5` result is reported as an alternative binary operationalization; do not imply a tested superiority of M5 merely because one p-value is below 0.05 and the other is not.